# Step 05. 판단 시나리오 1 — Cut-in / 선행차 감속

> **참고**: NGV_E2E_Education (Hanyang Univ.) — author Seokhwan Jeong (shjeong00@hanyang.ac.kr)  
> CARLA 0.9.16  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Python API로 **판단(Decision)을 시험할 시나리오**를 직접 구성합니다.  
선행차 급감속, 옆 차선 차량의 cut-in 상황을 재현합니다.

> ⚠️ **이 노트북의 CARLA 코드 셀은 시뮬레이터가 실행 중이어야 동작합니다.**  
> `./CarlaUE4.sh` 로 서버를 켠 뒤 위에서부터 실행하세요.

## 학습 목표

1. Waypoint 기반 차량 배치 (같은 차선 전/후)
2. 차량 속도 직접 제어로 시나리오 재현
3. 선행차 급감속 상황 구성
4. cut-in(끼어들기) 상황 구성

## 1. 환경 준비

### 이론 배경

시나리오는 **재현 가능**해야 합니다 → 동기 모드 + 고정 시드.  
정리용 `actors` 리스트에 spawn한 액터를 모아 마지막에 일괄 제거합니다.

In [ ]:
import carla, math

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
world = client.get_world()
traffic_manager = client.get_trafficmanager(8000)
tm_port = traffic_manager.get_port()
bp_lib = world.get_blueprint_library()
amap = world.get_map()
print('시나리오 환경 준비 (비동기). 실제 시나리오는 scripts/ 의 .py 로 실행')

## 2. Waypoint 기반 차량 배치

### 이론 배경

`map.get_waypoint(location)` 으로 도로 중심선의 waypoint를 얻고, `wp.next(dist)` 로 전방 지점을 구합니다.  
이렇게 하면 **같은 차선**에서 ego 뒤·lead 앞 같은 상대 배치를 정확히 만들 수 있습니다.  
(`z+0.3` 은 노면에 박히지 않게 살짝 띄움)

In [ ]:
start_wp = amap.get_waypoint(amap.get_spawn_points()[0].location)
ego_tf  = start_wp.transform
lead_tf = start_wp.next(20.0)[0].transform     # ego보다 20m 앞 (같은 차선)

ego_bp  = bp_lib.find('vehicle.tesla.model3'); ego_bp.set_attribute('role_name','ego')
lead_bp = bp_lib.find('vehicle.audi.tt');      lead_bp.set_attribute('role_name','lead')
ego  = world.try_spawn_actor(ego_bp,  carla.Transform(ego_tf.location  + carla.Location(z=0.3), ego_tf.rotation))
lead = world.try_spawn_actor(lead_bp, carla.Transform(lead_tf.location + carla.Location(z=0.3), lead_tf.rotation))
print('waypoint 기반 배치 확인 (ego, lead 20m 앞). 실제 거동은 위 스크립트로 실행')

## 3. 선행차 급감속

### 이론 배경

같은 차선 선행차를 `set_tm_target_speed()`로 정속 주행시키다가 목표 속도를 급격히 낮춰 감속을 유도합니다.  
Ego(autopilot)는 `distance_to_leading_vehicle` 설정에 따라 거리를 유지하며 감속해야 합니다.  
충돌 센서 + 최대 감속도로 반응을 평가합니다.

▶ 실행: `python3 scripts/run_scenario_decel.py`

In [ ]:
# 선행차 급감속은 정밀한 틱 타이밍이 필요 -> 스크립트로 실행
print('터미널에서 실행: python3 scripts/run_scenario_decel.py')
print('  ego(뒤) + lead(앞 20m) 배치 -> lead 40km/h 정속 후 급제동 -> ego(autopilot) 반응')

## 4. Cut-in 상황

### 이론 배경

인접 차선 차량을 **전방 거리 트리거**로 끼어들게 합니다. 핵심 API:

- `traffic_manager.force_lane_change(adv, to_right)` — Adversary를 Ego 차선으로 강제 차선 변경 (`set_path`보다 cut-in에 정확)
- `set_tm_target_speed()` — 제한속도 대비 %로 목표 속도 지정 (`vehicle_percentage_speed_difference`)
- 충돌 센서 + 최대 감속/최소 간격으로 **회피 성능 정량 평가**

cut-in은 TTC가 급격히 작아지는 대표 위험 상황입니다 → Step 06에서 정량 평가.

▶ 실행: `python3 scripts/run_scenario_cutin.py` (Town04, 결과/CSV 저장)

In [ ]:
# cut-in(끼어들기) 시나리오도 스크립트로 실행
print('터미널에서 실행: python3 scripts/run_scenario_cutin.py')
print('  왼쪽 차선 차량이 set_path로 ego 앞에 끼어듦 -> TTC 급감 (Step 06에서 정량 평가)')

## 5. 정리

### 이론 배경

시나리오 종료 후 액터를 제거하고 설정을 복구합니다.

In [ ]:
for a in [ego, lead]:
    if a is not None:
        try: a.destroy()
        except Exception: pass
print('시나리오 액터 정리 완료 (비동기)')